# Agentic AI Research Agent

This notebook is dededicated for developing a general-purpose research agent.

**Tech stack:** LangGraph `StateGraph`, LangChain `ChatOpenAI`, Tavily web search, SQLite checkpointing, and notebook-based execution.

**Workflow:** planner → initial web research → research brief → critic → optional follow-up research → revised brief.



## 1. Install dependencies

In [43]:
%pip install -U langgraph langgraph-checkpoint-sqlite langchain-openai langchain-core tavily-python python-dotenv


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Load API keys and imports



In [44]:
import os
import re
import sqlite3
from typing import TypedDict, List
from urllib.parse import urlsplit, urlunsplit

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tavily import TavilyClient

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.sqlite import SqliteSaver

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing. Add it to your .env file.")
if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY is missing. Add it to your .env file.")

model = ChatOpenAI(
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
    temperature=0
)
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

# Use a file-backed checkpoint database so thread state can persist across kernel restarts.
checkpoint_connection = sqlite3.connect(
    "research_agent_checkpoints.sqlite",
    check_same_thread=False
)
memory = SqliteSaver(checkpoint_connection)
memory.setup()


## 3. Define the graph state and structured query output

In [45]:
class AgentState(TypedDict, total=False):
    task: str
    plan: str
    draft: str
    critique: str
    content: List[str]
    revision_number: int
    max_revisions: int
    needs_more_research: bool
    missing_evidence: List[str]


class Queries(BaseModel):
    queries: List[str] = Field(
        description="Up to three distinct web search queries"
    )


class CritiqueResult(BaseModel):
    critique: str = Field(description="Concise assessment of the research brief")
    needs_more_research: bool = Field(
        description="Whether material evidence gaps remain"
    )
    missing_evidence: List[str] = Field(
        default_factory=list,
        description="Specific evidence gaps to investigate"
    )


## 4. Define the agent prompts

In [46]:
PLAN_PROMPT = """You are a research planning agent.
Break the user's question into a concise plan with the main sub-questions and evidence needed.
Do not write the final report yet. Keep the plan relevant to the user's exact question."""

RESEARCH_PLAN_PROMPT = """You are a web research agent.
Generate up to 3 focused and distinct search queries that cover different parts of the research plan.
Prefer credible, relevant sources and recent information when the question concerns a changing topic.
Return only the structured query list."""

WRITER_PROMPT = """You are a careful research synthesis agent.
Write a clear, balanced, structured research brief answering the user's question and following the research plan.

Rules:
- Base factual claims on the web evidence supplied below.
- Do not invent facts, source titles, dates, authors, or URLs.
- Add inline source citations such as [1] and [2] to claims supported by the evidence.
- End with a Sources section. Preserve each source's title and URL as supplied.
- Distinguish established evidence from interpretation.
- Mention meaningful uncertainty, conflicting findings, limitations, and unanswered questions.
- Address the previous critique if one is provided.

WEB EVIDENCE:
------
{content}
"""

REFLECTION_PROMPT = """You are a rigorous research critic.
Review the research brief against the question, plan, and supplied web evidence.

Check:
1. Whether the brief answers the question and covers the planned sub-questions.
2. Whether important factual claims have supporting citations.
3. Whether important evidence gaps or conflicting claims remain.
4. Whether the Sources section appears to correspond to the evidence supplied.

Return a structured result with a concise critique, needs_more_research, and specific missing_evidence.
Set needs_more_research=true only when another web search could materially address a gap.
Do not claim that a source has been independently verified when it has not."""

RESEARCH_CRITIQUE_PROMPT = """You are a follow-up research agent.
Based on the critic's feedback and named evidence gaps, generate up to 3 targeted search queries.
Prioritize verifying disputed claims, filling missing sub-questions, and finding authoritative sources.
Avoid repeating the same query or investigating issues that are only stylistic.
Return only the structured query list."""


## 5. Web search and source handling

In [57]:
def run_search(query: str, max_results: int = 3) -> List[str]:
    """Run Tavily search and retain source titles, URLs, dates, and snippets."""
    response = tavily.search(query=query, max_results=max_results)
    source_blocks = []

    for result in response.get("results", []):
        source_blocks.append(
            f"Title: {result.get('title', 'Untitled')}\n"
            f"URL: {result.get('url', '')}\n"
            f"Published date: {result.get('published_date') or 'not provided'}\n"
            f"Search relevance score: {result.get('score', 'not provided')}\n"
            f"Evidence snippet: {result.get('content', '')}"
        )

    return source_blocks


def canonical_url(source_block: str) -> str:
    """Normalize source URLs for basic duplicate detection."""
    match = re.search(r"(?m)^URL:\s*(\S+)", source_block)
    if not match:
        return ""
    raw_url = match.group(1).strip()

    try:
        parts = urlsplit(raw_url)
        host = (parts.netloc or "").lower()
        if host.startswith("www."):
            host = host[4:]
        path = parts.path.rstrip("/")
        return urlunsplit((parts.scheme.lower(), host, path, parts.query, ""))
    except Exception:
        return raw_url


def merge_unique_sources(existing: List[str], new_sources: List[str]) -> List[str]:
    """Keep sources unique by URL while preserving discovery order."""
    merged = []
    seen = set()

    for source in (existing or []) + (new_sources or []):
        if not isinstance(source, str) or not source.strip():
            continue
        key = canonical_url(source) or source.strip()
        if key in seen:
            continue
        seen.add(key)
        merged.append(source)

    return merged


def search_queries(queries):
    """Run Tavily searches and return normalized source records."""
    all_sources = []

    for query in queries:
        try:
            response = tavily.search(
                query=query,
                max_results=3
            )

            results = response.get("results", [])

            for item in results:
                url = item.get("url", "")
                if not url:
                    continue

                all_sources.append({
                    "query": query,
                    "title": item.get("title", "Untitled"),
                    "url": url,
                    "content": item.get("content", ""),
                    "score": item.get("score", 0.0)
                })

        except Exception as e:
            print(f"Search failed for {query!r}: {e}")

    # Remove duplicate URLs
    unique_sources = {}
    for source in all_sources:
        unique_sources[source["url"]] = source

    print(f"Retrieved {len(unique_sources)} unique sources.")
    return list(unique_sources.values())


def format_sources_for_prompt(sources: List[str]) -> str:
    """Number source blocks so inline citation markers can refer to them."""
    if not sources:
        return "No usable web evidence was retrieved."

    return "\n\n".join(
        f"[{index}]\n{source}"
        for index, source in enumerate(sources, start=1)
    )


## 6. Define the LangGraph nodes

In [49]:
def plan_node(state: AgentState):
    response = model.invoke([
        SystemMessage(content=PLAN_PROMPT),
        HumanMessage(content=state["task"])
    ])
    return {"plan": response.content}


def research_plan_node(state: AgentState):
    result = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_PLAN_PROMPT),
        HumanMessage(content=(
            f"Research question: {state['task']}\n"
            f"Research plan: {state.get('plan', '')}"
        ))
    ])

    # Handle either a dictionary or a Pydantic object
    queries = (
        result.get("queries", [])
        if isinstance(result, dict)
        else result.queries
    )

    new_sources = search_queries(queries)

    return {
        "content": merge_unique_sources(
            state.get("content", []),
            new_sources
        )
    }

def generation_node(state: AgentState):
    evidence = format_sources_for_prompt(state.get("content", []))
    user_message = HumanMessage(content=(
        f"Research question: {state['task']}\n\n"
        f"Research plan:\n{state.get('plan', '')}\n\n"
        f"Previous critique, if any:\n{state.get('critique', '')}\n\n"
        f"Evidence gaps, if any:\n{state.get('missing_evidence', [])}"
    ))

    response = model.invoke([
        SystemMessage(content=WRITER_PROMPT.format(content=evidence)),
        user_message
    ])

    return {
        "draft": response.content,
        "revision_number": state.get("revision_number", 0) + 1
    }


def reflection_node(state: AgentState):
    evidence = format_sources_for_prompt(state.get("content", []))

    result = model.with_structured_output(CritiqueResult).invoke([
        SystemMessage(content=REFLECTION_PROMPT),
        HumanMessage(content=(
            f"Research question: {state['task']}\n\n"
            f"Research plan:\n{state.get('plan', '')}\n\n"
            f"Current research brief:\n{state.get('draft', '')}\n\n"
            f"Sources and evidence:\n{evidence}"
        ))
    ])

    # Support both dictionary and Pydantic-object outputs.
    if isinstance(result, dict):
        critique = result.get("critique", "")
        needs_more_research = result.get("needs_more_research", False)
        missing_evidence = result.get("missing_evidence", [])
    else:
        critique = result.critique
        needs_more_research = result.needs_more_research
        missing_evidence = result.missing_evidence

    return {
        "critique": critique,
        "needs_more_research": needs_more_research,
        "missing_evidence": missing_evidence
    }



def research_critique_node(state: AgentState):
    result = model.with_structured_output(Queries).invoke([
        SystemMessage(content=RESEARCH_CRITIQUE_PROMPT),
        HumanMessage(content=(
            f"Research question: {state['task']}\n"
            f"Critique: {state.get('critique', '')}\n"
            f"Missing evidence: {state.get('missing_evidence', [])}"
        ))
    ])

    # Handle either a dictionary or a Pydantic object
    queries = (
        result.get("queries", [])
        if isinstance(result, dict)
        else result.queries
    )

    new_sources = search_queries(queries)

    return {
        "content": merge_unique_sources(
            state.get("content", []),
            new_sources
        )
    }

def route_after_critique(state: AgentState):
    """Stop when the critic is satisfied or the revision budget is reached."""
    revisions = state.get("revision_number", 0)
    max_revisions = state.get("max_revisions", 2)

    if revisions >= max_revisions:
        return END
    if state.get("needs_more_research", False):
        return "research_critique"
    return END


## 7. Build the graph

In [50]:
builder = StateGraph(AgentState)

builder.add_node("planner", plan_node)
builder.add_node("research_plan", research_plan_node)
builder.add_node("generate", generation_node)
builder.add_node("reflect", reflection_node)
builder.add_node("research_critique", research_critique_node)

builder.set_entry_point("planner")
builder.add_edge("planner", "research_plan")
builder.add_edge("research_plan", "generate")
builder.add_edge("generate", "reflect")

builder.add_conditional_edges(
    "reflect",
    route_after_critique,
    {
        END: END,
        "research_critique": "research_critique"
    }
)
builder.add_edge("research_critique", "generate")

graph = builder.compile(checkpointer=memory)


## 8. Run the research agent

### Example 1

In [51]:
thread = {"configurable": {"thread_id": "research-demo-1"}}

initial_state = {
    "task": (
        "Research recent advances in agentic AI, including key approaches, "
        "real-world applications, limitations, and open challenges."
    ),
    "content": [],
    "revision_number": 0,
    "max_revisions": 2,
    "needs_more_research": False,
    "missing_evidence": []
}

# This uses standard LangGraph node-update streaming.
# No get_stream_writer import or custom streaming API is required.
for update in graph.stream(
    initial_state,
    config=thread,
    stream_mode="updates"
):
    for node_name, node_update in update.items():
        print(f"\n--- {node_name} ---")

        if "plan" in node_update:
            print(node_update["plan"])
        elif "critique" in node_update:
            print(node_update["critique"])
            print("Needs more research:", node_update.get("needs_more_research"))
        elif "draft" in node_update:
            draft = node_update["draft"]
            print(draft[:1500] + ("..." if len(draft) > 1500 else ""))

final_state = graph.get_state(thread).values

print("\n\n===== FINAL RESEARCH BRIEF =====\n")
print(final_state.get("draft", "No research brief was produced."))



--- planner ---
### Research Plan for Recent Advances in Agentic AI

#### 1. Define Agentic AI
   - **Sub-questions:**
     - What is agentic AI and how is it distinguished from other forms of AI?
     - What are the key characteristics of agentic AI systems?
   - **Evidence Needed:**
     - Definitions and theoretical frameworks from recent literature.
     - Comparison with non-agentic AI systems.

#### 2. Key Approaches in Agentic AI
   - **Sub-questions:**
     - What are the main methodologies and frameworks used in developing agentic AI?
     - How do these approaches differ from traditional AI methods?
   - **Evidence Needed:**
     - Review of recent research papers and articles outlining methodologies.
     - Case studies highlighting innovative approaches in agentic AI.

#### 3. Real-World Applications
   - **Sub-questions:**
     - What are the current applications of agentic AI in various industries (e.g., healthcare, finance, robotics)?
     - How have these applications 

### Example 2

In [58]:

thread = {"configurable": {"thread_id": "research-demo-2"}}

initial_state = {
    "task": (
        "Research Retrieval-Augmented Generation (RAG) versus fine-tuning "
        "for Large Language Models (LLMs). Explain how both approaches work, "
        "compare their advantages and limitations, analyze real-world "
        "applications, discuss cost and scalability trade-offs, and identify "
        "when to choose one approach over the other. Support the findings "
        "with credible sources and practical examples."
    ),
    "content": [],
    "revision_number": 0,
    "max_revisions": 2,
    "needs_more_research": False,
    "missing_evidence": []
}

for update in graph.stream(
    initial_state,
    config=thread,
    stream_mode="updates"
):
    for node_name, node_update in update.items():
        print(f"\n--- {node_name} ---")

        if "plan" in node_update:
            print(node_update["plan"])
        elif "critique" in node_update:
            print(node_update["critique"])
            print(
                "Needs more research:",
                node_update.get("needs_more_research")
            )
        elif "draft" in node_update:
            draft = node_update["draft"]
            print(draft[:1500] + ("..." if len(draft) > 1500 else ""))

final_state = graph.get_state(thread).values

print("\n\n===== FINAL RESEARCH BRIEF =====\n")
print(final_state.get("draft", "No research brief was produced."))



--- planner ---
### Research Plan: Retrieval-Augmented Generation (RAG) vs. Fine-Tuning for Large Language Models (LLMs)

#### 1. **Understanding the Approaches**
   - **Sub-questions:**
     - What is Retrieval-Augmented Generation (RAG)?
     - What is fine-tuning in the context of Large Language Models (LLMs)?
   - **Evidence Needed:**
     - Definitions and mechanisms of RAG and fine-tuning.
     - Diagrams or flowcharts illustrating how each approach works.
     - Key papers or articles that describe these methodologies.

#### 2. **Comparative Analysis of Advantages and Limitations**
   - **Sub-questions:**
     - What are the advantages of RAG?
     - What are the limitations of RAG?
     - What are the advantages of fine-tuning?
     - What are the limitations of fine-tuning?
   - **Evidence Needed:**
     - Comparative studies or reviews that highlight strengths and weaknesses.
     - Case studies or examples demonstrating each approach's performance in various scenarios.

###

In [53]:

test_sources = search_queries([
    "recent advances in agentic AI research",
    "agentic AI research challenges and benchmarks"
])

print("\nNumber of sources:", len(test_sources))

for source in test_sources[:3]:
    print("\nTitle:", source["title"])
    print("URL:", source["url"])
    print("Content preview:", source["content"][:200])


Retrieved 6 unique sources.

Number of sources: 6

Title: Salesforce AI Research Advances Agentic Enterprise Readiness
URL: https://www.salesforce.com/uk/news/stories/ai-research-advances-enterprise-agentic-readiness
Content preview: This quarter, Salesforce AI Research unveiled an enterprise simulation environment to test agents’ ability to perform in realistic business scenarios, supported the launch of a new agentic benchmarkin

Title: Adobe Research » Adobe Research advances Agentic AI
URL: https://research.adobe.com/news/adobe-research-advances-agentic-ai
Content preview: These are our team’s key initiatives in our work on agentic AI:

 Model Development and Training: Creating and fine-tuning foundational models, including generating synthetic data that closely reflect

Title: How Riverside Research Advances Agentic AI for National Security - Riverside Research
URL: https://www.riversideresearch.org/insights/leaning-forward-how-riverside-research-advances-agentic-ai-for-national-s